Here will be a jupyter notebook for the FV analysis.

In [1]:
from cobra.flux_analysis import flux_variability_analysis
import jupyter_utils as ju

In [1]:
base_dir = Path.home() / "Documents" / "PhD" / "10-19 Research" / "11 Data" / "11.09_Models"
model = ju.load_model(base_dir,"Experiment", "250723_iABA974_BLG.sbml")

Run FVA

In [5]:
base_dir = Path.home() / "Documents" / "PhD" / "SoF1-GEM" / "Results" 
file_path = base_dir / "FVA_analysis" 

In [2]:
from cobra.flux_analysis import flux_variability_analysis
# Perform FVA for all reactions
fva_results = flux_variability_analysis(model, fraction_of_optimum=0.9)

# Display the results as a DataFrame
fva_df = fva_results.reset_index()
fva_df.columns = ["Reaction", "Minimum Flux", "Maximum Flux"]

base_dir = Path.home() / "Documents" / "PhD" / "SoF1-GEM" / "Data" 
file_path = base_dir / "FVA_analysis" 
#fva_df.to_excel(file_path/'250707_fva_autotrophic.xlsx')
print(fva_df)

: 

: 

Run FVA for a specific reaction

In [10]:
fva_results = flux_variability_analysis(model,reaction_list=('LINO_6_DESA'), fraction_of_optimum=0.9)
fva_df = fva_results.reset_index()
fva_df.columns = ["Reaction", "Minimum Flux", "Maximum Flux"]
print(fva_df)

      Reaction  Minimum Flux  Maximum Flux
0  LINO_6_DESA      0.000015      0.000017


Analyze FVA results

In [1]:
base_dir = Path.home() / "Documents" / "PhD" / "SoF1-GEM" 
file_path = base_dir / "Results" / "FVA_analysis" 
# Load the FVA result Excel file
fva_df = pd.read_excel(file_path/"250807_fva_BLG_autotrophic.xlsx")

# Optional: Display floats with 2 decimals
pd.options.display.float_format = "{:.2f}".format

# Add a flux span column
fva_df["span"] = fva_df["Maximum Flux"] - fva_df["Minimum Flux"]

# 1. Blocked reactions: both min and max = 0
blocked = fva_df[(fva_df["Minimum Flux"] == 0) & (fva_df["Maximum Flux"] == 0)]
num_blocked = len(blocked)

# 2. Essential reactions: min == max != 0
essential = fva_df[(fva_df["Minimum Flux"] == fva_df["Maximum Flux"]) & (fva_df["Minimum Flux"] != 0)]
num_essential = len(essential)

# 3. Flexible reactions: span > 0
flexible = fva_df[fva_df["span"] > 0]
num_flexible = len(flexible)

# 4. Highly variable reactions (span ≥ 100)
high_variability = fva_df[fva_df["span"] >= 100]
num_high_var = len(high_variability)

# 5. Top 10 most variable reactions
top_10_variable = fva_df.sort_values(by="span", ascending=False).head(10)

# Print summary
print("🔍 Summary of FVA Analysis")
print(f"Total reactions: {len(fva_df)}")
print(f"Blocked reactions: {num_blocked}")
print(f"Essential reactions (fixed flux): {num_essential}")
print(f"Flexible reactions (variable flux): {num_flexible}")
print(f"Highly variable (span ≥ 100): {num_high_var}")
print("\nTop 10 most variable reactions:")
print(top_10_variable[["Reaction", "Minimum Flux", "Maximum Flux", "span"]])

🔍 Summary of FVA Analysis
Total reactions: 2392
Blocked reactions: 892
Essential reactions (fixed flux): 0
Flexible reactions (variable flux): 1500
Highly variable (span ≥ 100): 300

Top 10 most variable reactions:
       Reaction  Minimum Flux  Maximum Flux    span
954    ECOAH5_1      -1000.00       1000.00 2000.00
1293      HACD2      -1000.00       1000.00 2000.00
1291    HACD1_1      -1000.00       1000.00 2000.00
932    ECOAH1_1      -1000.00       1000.00 2000.00
1311      HACD5      -1000.00       1000.00 2000.00
933      ECOAH2      -1000.00       1000.00 2000.00
1309      HACD4      -1000.00       1000.00 2000.00
2083  RHACOAR60      -1000.00       1000.00 2000.00
952      ECOAH4      -1000.00       1000.00 2000.00
1313      HACD6      -1000.00       1000.00 2000.00


In [2]:
fva_df['span'] = fva_df['span'].apply(lambda x: round(x, 6))

In [7]:
fva_df = pd.read_excel(file_path/"250709_fva_filtered_autotrophic.xlsx")
fva_df

,Unnamed: 0.1,Unnamed: 0,Reaction,Minimum Flux,Maximum Flux,span
0,0,924,ECOAH1,-1000.00,1000.00,2000.00
1,1,1098,G6PI,-1000.00,1000.00,2000.00
2,2,819,DADK,-1000.00,1000.00,2000.00
3,3,1139,GGGABADr,-1000.00,1000.00,2000.00
4,4,1140,GGGABADxr,-1000.00,1000.00,2000.00
...,...,...,...,...,...,...
1448,1448,164,EX_mobd_e,-0.00,-0.00,0.00
1449,1449,1058,FLVR,0.00,0.00,0.00
1450,1450,750,COabc,0.00,0.00,0.00
1451,1451,2380,THMP,0.00,0.00,0.00


In [6]:
fva_df_filtered= fva_df[fva_df['Maximum Flux'] != 0]
fva_df_filtered = fva_df_filtered.sort_values("span",ascending=False,ignore_index=True) # reorder by the span from the highest to the lowest
fva_df_filtered.to_excel(file_path/'250807_fva_filtered_BLG_autotrophic.xlsx')

In [ ]:
fva_df_filtered

remove loop reactions

In [ ]:
loop_reactions = [model.reactions.FRD7, model.reactions.SUCDi]
flux_variability_analysis(model, reaction_list=loop_reactions, loopless=False)